# Week 7 — Custom Dataset Training

The brief for this week is explicit about the dataset list, and Fashion-MNIST isn't on it — the point of Week 7 is to leave the MNIST family behind entirely and train on something with real photographic structure. I picked the **Smithsonian Butterflies** subset (`huggan/smithsonian_butterflies_subset` on Hugging Face Datasets): ~1000 RGB photos of butterfly specimens, resized to 64x64. It's small enough to train in a reasonable number of epochs on a single Colab GPU, but a genuine step up from grayscale 28x28 digits — three channels, real color and texture, no clean black background.

This is also the first notebook in the series where I can't actually run the cells myself. W&B logging needs a Weights & Biases account and API key, and the Hugging Face Hub push needs your own HF token — both of those are interactive logins that have to happen in your Colab session, not mine. So the `wandb.login()` and `huggingface_hub.login()` cells below are written to run as-is once you supply your own credentials; everything before and after those two cells should run unattended.

In [ ]:
!pip install -q datasets wandb huggingface_hub

In [ ]:
import math
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from datasets import load_dataset

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
torch.manual_seed(0)

RESOLUTION = 64
IMG_CHANNELS = 3

## Dataset and augmentation

I resize every image to 64x64 and center-crop to guard against non-square source photos, then apply a random horizontal flip — a flipped butterfly is still a perfectly plausible butterfly, since wing patterns are bilaterally symmetric by nature, so this is free extra data with no realism cost.

I deliberately did **not** add rotation or color jitter. Rotation would teach the model that butterflies can appear at arbitrary angles relative to the frame, which isn't true of how these specimen photos were actually taken (consistently oriented, wings spread, roughly centered) — training on rotated versions would just make the model worse at matching the real data distribution. Color jitter is even more directly harmful here: wing color and pattern *is* the signal that makes a generated image look like a specific butterfly species rather than a colorful blob, so deliberately corrupting color during training would actively fight against the thing I want the model to learn.

In [ ]:
transform = transforms.Compose([
    transforms.Resize(RESOLUTION),
    transforms.CenterCrop(RESOLUTION),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize([0.5] * IMG_CHANNELS, [0.5] * IMG_CHANNELS),
])


class ButterflyDataset(Dataset):
    def __init__(self, hf_dataset, transform):
        self.hf_dataset = hf_dataset
        self.transform = transform

    def __len__(self):
        return len(self.hf_dataset)

    def __getitem__(self, idx):
        image = self.hf_dataset[idx]["image"].convert("RGB")
        return self.transform(image)


hf_dataset = load_dataset("huggan/smithsonian_butterflies_subset", split="train")
train_dataset = ButterflyDataset(hf_dataset, transform)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)

print(f"{len(train_dataset)} training images, resized to {RESOLUTION}x{RESOLUTION}")

In [ ]:
class NoiseScheduler:
    def __init__(self, timesteps=1000, s=0.008, device=device):
        self.timesteps = timesteps
        steps = torch.arange(timesteps + 1, dtype=torch.float64) / timesteps
        f_t = torch.cos((steps + s) / (1 + s) * math.pi / 2) ** 2
        alphas_cumprod = f_t / f_t[0]
        alphas_cumprod = torch.clamp(alphas_cumprod, min=1e-9)

        self.alphas_cumprod = alphas_cumprod[1:].float().to(device)
        alphas_cumprod_prev = torch.cat([torch.tensor([1.0]), self.alphas_cumprod[:-1]])
        self.alphas_cumprod_prev = alphas_cumprod_prev.to(device)
        self.betas = (1 - self.alphas_cumprod / self.alphas_cumprod_prev).clamp(max=0.999)
        self.alphas = 1.0 - self.betas

    def add_noise(self, x0, t, noise=None):
        if noise is None:
            noise = torch.randn_like(x0)
        sqrt_ac = self.alphas_cumprod[t].sqrt().view(-1, 1, 1, 1)
        sqrt_one_minus_ac = (1 - self.alphas_cumprod[t]).sqrt().view(-1, 1, 1, 1)
        return sqrt_ac * x0 + sqrt_one_minus_ac * noise, noise

scheduler = NoiseScheduler(timesteps=1000, device=device)

## The model — unconditional, ported to 3 channels and 64x64

The Smithsonian Butterflies set doesn't come with class labels, so there's nothing to condition on this week — I'm reusing the plain unconditional `UNetDDPM` from Week 5 rather than Week 6's conditional version, just with `in_ch=3` instead of 1. The three down/up stages still work cleanly at 64x64: 64 → 32 → 16 → 8 spatial resolution through the encoder, mirrored back up through the decoder. Nothing about the architecture itself needed to change — only the input channel count and the resolution it's fed.

In [ ]:
class SinusoidalTimestepEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device).float() / half)
        args = t.float().unsqueeze(1) * freqs.unsqueeze(0)
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class TimestepMLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.embed = SinusoidalTimestepEmbedding(dim)
        self.mlp = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))

    def forward(self, t):
        return self.mlp(self.embed(t))


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.norm1 = nn.GroupNorm(min(8, in_ch), in_ch)
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.time_proj = nn.Linear(time_dim, out_ch)
        self.norm2 = nn.GroupNorm(min(8, out_ch), out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(F.silu(self.norm1(x)))
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(F.silu(self.norm2(h)))
        return h + self.skip(x)


class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.block = ResBlock(in_ch, out_ch, time_dim)
        self.pool = nn.Conv2d(out_ch, out_ch, 3, stride=2, padding=1)

    def forward(self, x, t_emb):
        h = self.block(x, t_emb)
        return self.pool(h), h


class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 2, stride=2)
        self.block = ResBlock(in_ch + skip_ch, out_ch, time_dim)

    def forward(self, x, skip, t_emb):
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.pad(x, (0, skip.shape[-1] - x.shape[-1], 0, skip.shape[-2] - x.shape[-2]))
        x = torch.cat([x, skip], dim=1)
        return self.block(x, t_emb)


class UNetDDPM(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, time_dim=128):
        super().__init__()
        self.time_mlp = TimestepMLP(time_dim)
        self.inc = ResBlock(in_ch, base_ch, time_dim)
        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)
        self.down3 = Down(base_ch * 4, base_ch * 8, time_dim)
        self.bottleneck = ResBlock(base_ch * 8, base_ch * 8, time_dim)
        self.up1 = Up(base_ch * 8, base_ch * 8, base_ch * 4, time_dim)
        self.up2 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        self.up3 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)
        self.outc = nn.Conv2d(base_ch, in_ch, 1)

    def forward(self, x, t):
        t_emb = self.time_mlp(t)
        h0 = self.inc(x, t_emb)
        h1, skip1 = self.down1(h0, t_emb)
        h2, skip2 = self.down2(h1, t_emb)
        h3, skip3 = self.down3(h2, t_emb)
        h3 = self.bottleneck(h3, t_emb)
        h = self.up1(h3, skip3, t_emb)
        h = self.up2(h, skip2, t_emb)
        h = self.up3(h, skip1, t_emb)
        return self.outc(h)

model = UNetDDPM(in_ch=IMG_CHANNELS).to(device)
print(sum(p.numel() for p in model.parameters()), "parameters")

## Weights & Biases logging

The brief asks for W&B tracking of losses, sample images, and hyperparameters. `wandb.login()` is interactive — it'll prompt for an API key the first time you run this cell in your own Colab session (free account at wandb.ai). Everything downstream logs to the run this cell creates: per-epoch loss and a sample grid at each training milestone.

In [ ]:
import wandb

wandb.login(key=os.environ.get("WANDB_API_KEY"))

EPOCHS = 100
BATCH_SIZE = 32
LR = 2e-4

run = wandb.init(
    project="diffusion-models-from-scratch",
    name="week7-butterflies-ddpm",
    config={
        "dataset": "huggan/smithsonian_butterflies_subset",
        "resolution": RESOLUTION,
        "batch_size": BATCH_SIZE,
        "epochs": EPOCHS,
        "lr": LR,
        "base_ch": 64,
    },
)

## DDIM sampling

Same deterministic DDIM sampler from Week 5, unconditional, adapted to a `(N, 3, 64, 64)` shape. I use this instead of the full 1000-step DDPM sampler purely to keep milestone-sample generation fast during training — DDPM would also work, just slower to call repeatedly.

In [ ]:
@torch.no_grad()
def ddim_sample(model, scheduler, shape, num_steps=50, eta=0.0, device=device):
    model.eval()
    T = scheduler.timesteps

    step_indices = torch.linspace(0, T - 1, num_steps).long().flip(0).to(device)
    x = torch.randn(shape, device=device)
    ac = scheduler.alphas_cumprod

    for i in range(len(step_indices)):
        t = step_indices[i]
        t_batch = torch.full((shape[0],), t.item(), device=device, dtype=torch.long)

        ac_t = ac[t]
        ac_prev = ac[step_indices[i + 1]] if i + 1 < len(step_indices) else torch.tensor(1.0, device=device)

        eps = model(x, t_batch)
        x0_pred = ((x - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()).clamp(-1, 1)
        dir_coeff = torch.sqrt((1 - ac_prev).clamp(min=0))
        x = ac_prev.sqrt() * x0_pred + dir_coeff * eps

    model.train()
    return x.clamp(-1, 1)

## Training: 100 epochs, milestones every 20

This is the part the README warns will take a while — 100 epochs over ~1000 64x64 RGB images on a T4 is a real training run, not a quick demo, so expect this cell to run for an extended stretch. I save a checkpoint and a sample grid (both locally to `samples/` and to W&B) every 20 epochs (20/40/60/80/100), which is what the deliverable's "samples/ folder with generations at multiple training milestones (every 10-20 epochs)" asks for. Checkpointing at every milestone also means a Colab disconnect partway through doesn't lose everything — I can resume from the last saved `.pt` file instead of restarting from epoch 0. A `TRAINING_LOG.md` documenting the run gets written at the end.

In [ ]:
os.makedirs("samples", exist_ok=True)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
milestones = {20, 40, 60, 80, 100}
losses = []

for epoch in range(1, EPOCHS + 1):
    running_loss = 0.0
    n_samples = 0
    for x0 in train_loader:
        x0 = x0.to(device)
        t = torch.randint(0, scheduler.timesteps, (x0.shape[0],), device=device)
        xt, noise = scheduler.add_noise(x0, t)
        pred_noise = model(xt, t)
        loss = F.mse_loss(pred_noise, noise)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        running_loss += loss.item() * x0.shape[0]
        n_samples += x0.shape[0]

    epoch_loss = running_loss / n_samples
    losses.append(epoch_loss)
    wandb.log({"epoch": epoch, "loss": epoch_loss})
    print(f"epoch {epoch}/{EPOCHS}  loss={epoch_loss:.4f}")

    if epoch in milestones:
        samples = ddim_sample(model, scheduler, (16, IMG_CHANNELS, RESOLUTION, RESOLUTION), num_steps=50)
        grid = make_grid(samples, nrow=4, normalize=True, value_range=(-1, 1))

        # mode-collapse check: if every sample in the grid looks near-identical, that's the symptom
        grid_path = f"samples/samples_epoch_{epoch}.png"
        plt.figure(figsize=(6, 6))
        plt.imshow(grid.permute(1, 2, 0).cpu())
        plt.axis("off")
        plt.title(f"epoch {epoch}")
        plt.savefig(grid_path, bbox_inches="tight")
        plt.show()

        wandb.log({"samples": wandb.Image(grid_path), "epoch": epoch})

        # checkpoint every milestone so a Colab disconnect only costs up to 20 epochs of progress,
        # not the whole run -- resume by loading the latest butterfly_ddpm_epoch_*.pt into `model`
        ckpt_path = f"butterfly_ddpm_epoch_{epoch}.pt"
        torch.save(model.state_dict(), ckpt_path)
        print(f"saved checkpoint to {ckpt_path}")

plt.figure()
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("MSE loss")
plt.title("Training loss")
plt.savefig("samples/training_curves.png", bbox_inches="tight")
plt.show()

training_log = f"""# Training Log — Week 7 Butterflies DDPM

## Setup
- Dataset: huggan/smithsonian_butterflies_subset, {len(train_dataset)} images, resized to {RESOLUTION}x{RESOLUTION}
- Batch size: {BATCH_SIZE}, learning rate: {LR}, epochs: {EPOCHS}
- Augmentation: random horizontal flip only (see markdown above for why rotation/color jitter were excluded)

## Decisions made
- Chose Smithsonian Butterflies over Pokemon/anime-faces/CelebA for its small size (~1000 images),
  which keeps a 100-epoch run inside a single Colab session.
- Kept the model unconditional (no class labels) since this dataset has no per-image class structure
  to condition on.
- Checkpointed every 20 epochs specifically so a Colab disconnect never costs more than 20 epochs
  of progress.

## What to watch for when reviewing this run
- Mode collapse: if the epoch-100 sample grid shows near-identical butterflies (same pose/color
  repeated across all 16 samples), that's the symptom -- check the milestone grids in `samples/`
  for whether diversity holds up as training progresses, not just whether quality improves.
- Loss vs. visual quality: diffusion loss can keep dropping while samples plateau visually -- the
  milestone grids are the real signal, not just the final loss number in the curve above.

## Final loss
{losses[-1]:.4f} at epoch {EPOCHS} (see samples/training_curves.png for the full curve)
"""

with open("TRAINING_LOG.md", "w", encoding="utf-8") as f:
    f.write(training_log)
print("\nWrote TRAINING_LOG.md")

## Sample gallery across milestones

Side by side comparison of the epoch-25, epoch-50, and epoch-100 sample grids saved above, so the progression from rough blobs to recognizable butterfly shapes is visible in one place.

In [ ]:
sorted_milestones = sorted(milestones)
fig, axes = plt.subplots(1, len(sorted_milestones), figsize=(6 * len(sorted_milestones), 6))
for ax, epoch in zip(axes, sorted_milestones):
    img = plt.imread(f"samples/samples_epoch_{epoch}.png")
    ax.imshow(img)
    ax.set_title(f"epoch {epoch}")
    ax.axis("off")
plt.tight_layout()
plt.show()

## Pushing the checkpoint to Hugging Face Hub

`login()` here is the second interactive step — it needs your own Hugging Face access token (from huggingface.co/settings/tokens, write access). Replace `HF_USERNAME` below with your actual username before running.

In [ ]:
from huggingface_hub import HfApi, create_repo, login

login(token=os.environ.get("HF_TOKEN"))

HF_USERNAME = os.environ.get("HF_USERNAME", "IshaanM05")
REPO_ID = f"{HF_USERNAME}/butterfly-ddpm-week7"

create_repo(REPO_ID, exist_ok=True)

api = HfApi()
api.upload_file(
    path_or_fileobj=f"butterfly_ddpm_epoch_{EPOCHS}.pt",
    path_in_repo="butterfly_ddpm.pt",
    repo_id=REPO_ID,
)
print(f"Checkpoint pushed to https://huggingface.co/{REPO_ID}")

wandb.finish()

## Self-check questions

**1. What was the hardest decision you made about your dataset?**

Honestly, the hardest call was ruling out Fashion-MNIST as a lazy continuation of Weeks 4-6. It would have been the path of least resistance — same shape, same architecture, zero new bugs — but the whole point of this week is leaving the MNIST family behind and dealing with real photographic data. Between the remaining options, I picked Smithsonian Butterflies over CelebA or anime faces specifically because of its size: ~1000 images is small enough to get through 100 epochs in a single Colab session without an overnight run, while still being genuinely harder than digits (RGB, real texture, no clean background).

**2. What augmentations did you use and why? Which would you NOT use (and why)?**

Just a random horizontal flip. Butterfly wings are bilaterally symmetric, so a flipped photo is still a completely plausible butterfly — that's free additional training variation with no realism cost. I deliberately left out rotation and color jitter. Rotation would teach the model that butterflies appear at arbitrary angles, which doesn't match how these specimen photos were actually taken (consistently oriented, wings spread, centered in frame) — it would make the model worse at matching the true data distribution, not better. Color jitter is worse still: wing color and pattern is the actual signal that distinguishes one butterfly from a colorful blob, so deliberately perturbing color during training fights directly against the thing the model needs to learn.

**3. What batch size + learning rate worked? How did you choose?**

Batch size 32 at 64x64x3 — large enough to get a reasonably stable gradient estimate without running into T4 memory limits with this UNet's channel counts (up to 512 at the bottleneck). Learning rate 2e-4 is the same value that worked fine for the MNIST runs in Weeks 4-6; I didn't see a strong reason to retune it going in, since AdamW at this LR is a pretty standard, forgiving starting point for small diffusion UNets, and I'd rather spend the epoch budget on training than on a learning-rate sweep for a single custom-dataset run.

**4. What would you change if you had 10x the compute?**

Three things, in order of expected impact: first, an EMA (exponential moving average) of the model weights — it's close to free compute-wise and tends to visibly clean up sample quality, so there's no real reason not to have it with more budget to spare. Second, I'd push past 64x64 to 128x128 or higher, since 64x64 genuinely caps how much wing detail the model can ever represent regardless of training time. Third, I'd just run more epochs and watch the loss curve longer — 100 epochs is the brief's minimum, not necessarily the point of diminishing returns, and with 10x compute I'd rather verify where that point actually is than guess.

**5. How did you decide on image resolution? What trade-offs did you consider?**

64x64 was really the README's own suggestion for every dataset on the approved list, but it's also the right trade-off point for this specific run: high enough that butterfly wing patterns are still recognizable (28x28 would flatten most of that detail away), low enough that a 100-epoch run with this UNet's channel counts (up to 512 at the bottleneck) finishes in a reasonable amount of time on a single T4. Going to 128x128 would roughly quadruple the compute per step for marginal detail gain on a dataset this small (~1000 images) — not a trade worth making until the lower resolution is clearly working.

**6. What signs of mode collapse did you watch for?**

The main signal is in the milestone sample grids themselves: if the 16 samples at a given epoch all show the same pose, color palette, or wing shape with no meaningful variation between them, that's mode collapse — the model has found one or two "safe" outputs that minimize loss on average rather than actually modeling the full diversity of the training set. I specifically compare grids *across* milestones (20 vs 60 vs 100), not just at the final epoch, since collapse that creeps in partway through training is easy to miss if you only ever look at the last checkpoint.

**7. How did you decide when to stop training?**

I used the brief's stated minimum of 100 epochs as the actual stopping point here, but the real signal I'd use to validate that choice is whether the loss curve has visibly plateaued by the end (see `samples/training_curves.png`) — if it's still dropping at epoch 100, that's a sign more epochs would likely keep helping; if it's flattened out, 100 was a reasonable place to stop. I'd weigh that against the milestone sample grids too: if epoch 80 and epoch 100 look about equally good, that's corroborating evidence training has converged rather than just the loss number alone, since (per the debugging notes above) loss and visual quality don't always move together.